# 04 · Your First Live Bot (rules only)

We put everything together:
1. **Ask** Telegram for new messages (`getUpdates`)
2. **Decide** what to reply (`reply()` — your function from notebook 01)
3. **Send** the reply (`sendMessage`)
4. Repeat forever 🔁

This is called **polling**.

In [ ]:
import os
from getpass import getpass

import requests
from dotenv import load_dotenv

load_dotenv()
TOKEN = os.environ.get("TELEGRAM_BOT_TOKEN") or getpass("Paste your bot token: ")
BASE = f"https://api.telegram.org/bot{TOKEN}"

## 1. The brain — your rules

In [ ]:
prices = {
    "kopi": 1.40,
    "kopi-o": 1.20,
    "kopi-c": 1.50,
    "teh": 1.40,
    "teh-o": 1.20,
    "teh-c": 1.50,
    "milo": 1.80,
}


def reply(text, name):
    text = text.lower().strip()

    if text in ["/start", "hi", "hello"]:
        return f"Hello {name}! Uncle here. Type 'menu' to see what we have."

    if text == "menu":
        lines = [f"{drink.title()}: ${price:.2f}" for drink, price in prices.items()]
        return "Today's menu:\n" + "\n".join(lines)

    # check longer names first, so "kopi-o" isn't mistaken for "kopi"
    for drink in sorted(prices, key=len, reverse=True):
        if drink in text:
            return f"One {drink.title()} coming up! That's ${prices[drink]:.2f}."

    return "Aiyo, Uncle don't understand. Type 'menu' lah."


# test the brain before going live
print(reply("menu", "Mei Ling"))
print(reply("one kopi-o please", "Mei Ling"))

## 2. Send and receive

In [ ]:
def get_updates(offset):
    # timeout 30 = Telegram waits up to 30 seconds for a new message
    params = {"offset": offset, "timeout": 30}
    return requests.get(f"{BASE}/getUpdates", params=params).json()["result"]


def send_message(chat_id, text):
    requests.post(f"{BASE}/sendMessage", json={"chat_id": chat_id, "text": text})

## 3. Go live! 🚀
Run the cell below, then chat with your bot on Telegram.

Press the **■ Stop** button (Interrupt) at the top of the notebook to stop it.

In [ ]:
offset = None   # which update we've already handled
print("Bot is running... message it on Telegram!")

while True:
    for update in get_updates(offset):
        offset = update["update_id"] + 1   # don't handle this one again

        message = update.get("message", {})
        if "text" not in message:
            continue   # skip stickers, photos...

        chat_id = message["chat"]["id"]
        name = message["from"]["first_name"]
        text = message["text"]

        answer = reply(text, name)
        send_message(chat_id, answer)
        print(f"{name}: {text}  ->  {answer}")

> 🛑 When you press Stop you'll see a red **KeyboardInterrupt** message. That's normal — it just means *"you stopped me"*.

## 🤔 Try to break it
Ask your neighbour to message your bot:
- *"can I have a kopi but less sweet?"*
- *"2 teh and 1 milo, how much?"*
- *"is it raining?"*

How many `if` statements would you need to handle **everything** people might say? 😅

**Tomorrow we replace `reply()` with an AI agent** — the loop stays exactly the same.

## 🖥️ Run it as a file
Everything above is also saved in **`day1_bot.py`** — one file you can run without the notebook.
Stop the notebook bot first (■), then open a **Terminal** (Ctrl + `) and type:
```
python day1_bot.py
```
Press **Ctrl + C** to stop it.

> 💡 `day1_bot.py` has the same code as this notebook, plus a small **web app** (part 5) so it can also run online. That's next!

## ✏️ Exercises
1. Add **"yuan yang"** and **"milo dinosaur"** to `prices`.
2. If the message contains **"peng"**, add $0.30 and say *"iced one ah!"*
3. Make `/help` list everything the bot understands.

---
# 🌍 Put your bot online (webhook)

Right now your bot only works while Codespaces is running, because **we** keep asking Telegram for messages (**polling**).

Online, it works the other way round: Telegram **sends** each message to our web app (a **webhook**).
```
Polling:  your code  --"any new messages?"-->  Telegram     (python day1_bot.py)
Webhook:  Telegram   --"here's a message!"-->  your web app (Vercel)
```
⚠️ A bot can only use **one** at a time. Running `python day1_bot.py` switches it back to polling.

## Step 1 — Deploy to Vercel
1. **Source Control** tab → type a message → **Commit** → **Sync Changes** (saves your code to GitHub).
2. **vercel.com** → **Add New → Project** → import your repo → **Deploy**.
   (`pyproject.toml` tells Vercel to run `day1_bot.py`.)
3. **Settings → Environment Variables** → add `TELEGRAM_BOT_TOKEN`, and `WEBHOOK_SECRET` (make up any password — add the same line to your `.env`).
4. **Deployments → ⋯ → Redeploy** (new settings only apply after a redeploy).

## Step 2 — Is it online?

In [ ]:
APP_URL = "https://YOUR-APP.vercel.app"   # 👈 paste YOUR Vercel URL here

requests.get(APP_URL).json()

You should see `{'status': 'Kopi Uncle is running!'}`.

## Step 3 — Tell Telegram where your bot lives 🔔
Stop the notebook bot and `python day1_bot.py` first, then:

In [ ]:
load_dotenv()
WEBHOOK_SECRET = os.environ.get("WEBHOOK_SECRET")   # the same password you put in Vercel

response = requests.post(f"{BASE}/setWebhook", json={
    "url": APP_URL + "/webhook",
    "secret_token": WEBHOOK_SECRET,
})
response.json()

`'ok': True`? 🎉 Close your laptop — your bot still replies!

Tomorrow you'll give it an AI brain and push again. Vercel updates by itself, and Telegram keeps sending messages to the same address.